# 01: Public datasets

Design decisions
- Four public tables, each its own task with its own recorded outcome: BRFSS 2015 heart disease indicators (Kaggle, alexteboul; outcome HeartDiseaseorAttack, the CDC variable _MICHD), CDC Diabetes Health Indicators (UCI id 891; outcome Diabetes_binary), UCI Heart Disease from four sites (UCI id 45; outcome num > 0) and Cardiovascular Disease 70k (Kaggle, sulianova; outcome cardio). Tables are never pooled across outcomes.
- Raw files are written to `datasets/raw/<name>/` on Drive, outside the repository; each file's SHA-256 is recorded. Kaggle downloads read `kaggle.json` from the Drive project folder.
- Checks: published row counts for every table and site; known positive counts for BRFSS heart disease (23,893) and Cardio (34,979); missing values counted.
- The BRFSS heart table and the CDC Diabetes table come from the same BRFSS 2015 respondents. Their shared predictors are compared row by row, and the published Diabetes_binary is cross-tabulated against the heart table's three-level Diabetes variable to record how prediabetes is coded.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ucimlrepo', 'kaggle==1.6.17'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public

SEED = 42
seed_everything(SEED)
paths.ensure_dirs()
print('raw data folder:', paths.RAW)

## Downloads (skipped when the file is already on Drive)

In [ ]:
FILES = {
    'brfss2015_heart': public.download_kaggle('brfss2015_heart', paths.RAW, paths.DRIVE_ROOT),
    'cardio70k': public.download_kaggle('cardio70k', paths.RAW, paths.DRIVE_ROOT),
    'cdc_diabetes': public.download_cdc_diabetes(paths.RAW),
}
for site, f in public.download_uci_heart(paths.RAW).items():
    FILES[f'uci_heart_{site}'] = f
SHA = {name: sha256_file(f) for name, f in FILES.items()}
for name, f in FILES.items():
    print(f'{name:22s} {f}  sha256 {SHA[name][:16]}...')

## Dataset cards and checks

In [ ]:
CARDS = pd.DataFrame([public.card(name, paths.RAW) for name in public.SPECS])
print(CARDS[['dataset', 'rows', 'expected_rows', 'predictors', 'positives', 'prevalence', 'missing_values']].to_string(index=False))

assert (CARDS['rows'] == CARDS['expected_rows']).all(), 'row counts differ from the published sizes'
cards = CARDS.set_index('dataset')
assert int(cards.loc['brfss2015_heart', 'positives']) == 23893
assert int(cards.loc['cardio70k', 'positives']) == 34979
assert int(cards.loc['brfss2015_heart', 'missing_values']) == 0
assert int(cards.loc['cdc_diabetes', 'missing_values']) == 0
Xu, yu = public.load_uci_heart(paths.RAW)
SITE_ROWS = Xu['site'].value_counts().to_dict()
assert SITE_ROWS == public.UCI_HEART_SITES, SITE_ROWS
SITE_PREVALENCE = pd.Series(yu).groupby(Xu['site'].to_numpy()).mean().round(4).to_dict()
print('UCI rows per site:', SITE_ROWS)
print('UCI prevalence per site:', SITE_PREVALENCE)

## Same respondents in the BRFSS heart and CDC Diabetes tables; coding of Diabetes_binary

In [ ]:
heart = pd.read_csv(FILES['brfss2015_heart'])
diab = pd.read_csv(FILES['cdc_diabetes'])
shared = [c for c in public.BRFSS_FEATURES if c != 'Diabetes'] + ['HeartDiseaseorAttack']
same_order = bool(len(heart) == len(diab) and (heart[shared].to_numpy() == diab[shared].to_numpy()).all())
same_sums = bool((heart[shared].sum() == diab[shared].sum()).all())
if same_order:
    CODING = pd.crosstab(heart['Diabetes'], diab['Diabetes_binary'])
    print(CODING)
    coding = {f'Diabetes={int(i)}': {f'Diabetes_binary={int(j)}': int(CODING.loc[i, j]) for j in CODING.columns}
              for i in CODING.index}
    prediabetes_coded_as = int(CODING.loc[1.0].idxmax()) if 1.0 in CODING.index else None
    method = 'row-by-row crosstab'
else:
    coding = None
    positives = int(diab['Diabetes_binary'].sum())
    if same_sums and positives == int((heart['Diabetes'] == 2).sum()):
        prediabetes_coded_as, method = 0, 'positive count equals Diabetes == 2 count'
    elif same_sums and positives == int((heart['Diabetes'] >= 1).sum()):
        prediabetes_coded_as, method = 1, 'positive count equals Diabetes >= 1 count'
    else:
        prediabetes_coded_as, method = None, 'undetermined'
PROVENANCE = {'shared_predictors_identical_row_by_row': same_order,
              'shared_predictor_sums_identical': same_sums,
              'diabetes_binary_crosstab': coding,
              'diabetes_binary_positives': int(diab['Diabetes_binary'].sum()),
              'heart_table_diabetes_level_counts': {f'Diabetes={int(k)}': int(v) for k, v in heart['Diabetes'].value_counts().sort_index().items()},
              'prediabetes_coded_as': prediabetes_coded_as,
              'coding_method': method}
print(json.dumps(PROVENANCE, indent=1))

In [ ]:
CARDS.to_csv(f'{paths.RESULTS}/01_dataset_cards.csv', index=False)
write_json(f'{paths.RESULTS}/01_dataset_cards.json',
           {'seed': SEED, 'cards': CARDS.to_dict('records'), 'sha256': SHA,
            'uci_rows_per_site': SITE_ROWS, 'uci_prevalence_per_site': SITE_PREVALENCE,
            'brfss_heart_and_cdc_diabetes': PROVENANCE})
print('written: results/01_dataset_cards.csv, results/01_dataset_cards.json')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "01: public datasets downloaded and checked (dataset cards, file hashes, shared-respondent and diabetes-coding checks)"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)